# Real-data leakage audit: complete walkthrough

Install `python -m pip install -e ".[dev]"` and optionally `jupyterlab matplotlib`. Select that environment as the kernel. This deterministic toolkit has no model to train. We acquire unchanged, checksum-pinned UCI Adult observations, inspect aggregate quality signals, run production audits, compare independent references, and save a new report. Cells are deliberately unexecuted in Git; running them does not invent new benchmark results.

In [ ]:
from pathlib import Path
import os, sys, json, hashlib
from datetime import datetime, timezone
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), 'Open this notebook from its repository.'
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT))
RUN_REAL_PIPELINE = False  # Set True to download/process real data and create fresh artifacts.
RUN = ROOT / 'data' / 'interim' / ('notebook-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
print('Repository:', ROOT, '\nFresh run:', RUN)


## Recorded evidence and provenance
These are previously executed reports, not results of this notebook. UCI Adult: Becker & Kohavi, CC BY 4.0, DOI 10.24432/C5XW20. Near matches are review signals; there is no identity or harmful-leakage ground truth.

In [ ]:
recorded = json.loads((ROOT / 'docs/near-duplicate-results.json').read_text())
print(json.dumps(recorded, indent=2))

## Download and validate original observations
Enable the switch above to execute. Production preparation verifies archive/member hashes and keeps official splits. All generated artifacts stay under ignored data/. No synthetic rows are added.

In [ ]:
from leakage_audit.adult_validation import validate as validate_adult
if RUN_REAL_PIPELINE:
    RUN.mkdir(parents=True, exist_ok=False)
    adult_report = validate_adult(report=RUN / 'adult.json')
else:
    print('Download/audit skipped; enable RUN_REAL_PIPELINE to run.')

## Inspect aggregate schema and missingness
Raw person-level rows are not displayed or embedded in the notebook.

In [ ]:
from leakage_audit import read_csv, audit, AuditConfig
if RUN_REAL_PIPELINE:
    train = read_csv(ROOT / 'data/processed/adult/train.csv')
    test = read_csv(ROOT / 'data/processed/adult/test.csv')
    print({'train_rows': len(train), 'test_rows': len(test), 'columns': list(train[0])})
    missing = {c: sum(r[c] in ('', None) for r in train) for c in train[0]}
    print('Training missing counts:', missing)

## Production audit and independent near-match verification
Target-quality checks are explicit. Default feature-duplicate comparisons exclude target labels. The independent sorted-group oracle verifies exact one-feature near-match counts.

In [ ]:
from leakage_audit.near_validation import validate as validate_near
if RUN_REAL_PIPELINE:
    result = audit(train, test, AuditConfig('target', check_near_duplicates=True))
    (RUN / 'production-audit.json').write_text(json.dumps(result, indent=2, allow_nan=False))
    verified = validate_near(report=RUN / 'near-verified.json')
    assert verified['independent_sorted_reference_agrees']
    print('Independent verification passed; reports:', RUN)

## Visualize recorded aggregate findings
Counts per differing column can overlap; do not sum them as distinct people. Survey-weight differences dominate. The plotted report is explicitly historical unless the pipeline ran above.

In [ ]:
plot_report = verified if RUN_REAL_PIPELINE else recorded
counts = plot_report['near_matches']['differing_column_counts']
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.barh(list(counts), list(counts.values()))
    ax.set_xlabel('Heldout feature vectors matching except this column')
    ax.set_title('Real Adult near-match diagnostics' + (' — fresh run' if RUN_REAL_PIPELINE else ' — recorded evidence'))
    plt.tight_layout()
except ImportError:
    print('Install matplotlib to render; aggregate counts:', counts)

## Interpretation and reproducibility
Review collection provenance and deployment availability before acting on findings. Neither duplicates nor distribution shift alone prove harmful leakage. Read `Report.md`, `docs/near-duplicates.md`, and `docs/methodology.md`. Run `python -m pytest -q` for mechanics; these tests are separate from real-data validation. Commit source and aggregate reports only, never raw data or notebook outputs containing records.